In [1]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.io
from pathlib import Path
import sys
import time
from sklearn.linear_model import LogisticRegression
from sklearn import tree
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import importlib
import pandas as pd

C:\Users\auder\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\auder\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
dossier_actuel = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
racine_projet = dossier_actuel.parent
sys.path.append(str(racine_projet))
%load_ext autoreload
%autoreload 2
import scripts.functions as fc

In [3]:
Path_data = racine_projet/"data"/"ECGData.mat"
Path_atome = racine_projet/"data"/"Atomes.mat"
X,y_classic,signaux,D,Z = fc.Import_Data(Path_data,Path_atome)
_,y_chf,_,_,_=fc.Import_Data(Path_data,Path_atome,Sort='chf')
_,y_nsr1,_,_,_=fc.Import_Data(Path_data,Path_atome,Sort='nsr1')
_,y_nsr2,_,_,_=fc.Import_Data(Path_data,Path_atome,Sort='nsr2')

In [4]:
Models=[LogisticRegression(max_iter=10**9),tree.DecisionTreeClassifier(criterion='entropy'),RandomForestClassifier(),GaussianNB(),BernoulliNB(),
        KNeighborsClassifier(n_neighbors=10),KNeighborsClassifier(n_neighbors=11),KNeighborsClassifier(n_neighbors=12)]
ARR=96
CHF=ARR+30
Y=[y_classic,y_chf,y_nsr1,y_nsr2]
Sort=['classic','chf','nsr1','nsr2']

## Naive Classification :

In [28]:
importlib.reload(fc)

<module 'scripts.functions' from 'C:\\Users\\auder\\Documents\\Université\\L3\\Stage_ECG_git\\stage_ecg\\code\\scripts\\functions.py'>

In [9]:
for model in Models :
    tic = time.time()
    err,prtable=fc.Classifier_eq_iter(signaux,y_classic,model)
    print(f'ERR_{model}_Raw_100 :',err)
    print(prtable)
    print(f'time {model}_Raw_100 :',time.time()-tic)

ERR_LogisticRegression(max_iter=1000000000)_Raw_100 : 0.4190909090909091
   Precision     Recall   F1_Score
A  61.140007  77.336842  68.052409
C  50.248016  28.214286  33.760306
N  53.183622  33.732143  39.084705
time LogisticRegression(max_iter=1000000000)_Raw_100 : 408.5701570510864
ERR_DecisionTreeClassifier(criterion='entropy')_Raw_100 : 0.4200000000000001
   Precision     Recall   F1_Score
A  67.267522  70.184211  68.261014
C  31.911760  29.233333  29.273417
N  56.156183  50.089286  50.986390
time DecisionTreeClassifier(criterion='entropy')_Raw_100 : 2375.7177588939667
ERR_RandomForestClassifier()_Raw_100 : 0.28090909090909094
   Precision     Recall   F1_Score
A  68.065029  98.278947  80.354305
C  14.000000   2.366667   4.047619
N  95.499603  60.928571  72.850653
time RandomForestClassifier()_Raw_100 : 365.09498858451843
ERR_GaussianNB()_Raw_100 : 0.39454545454545453
   Precision     Recall   F1_Score
A  64.617880  76.094737  69.674265
C  54.233333  10.680952  17.335065
N  51.316

## Activation time classification

In [11]:
for i in range(len(Y)) :# On classifie selon chacun des modes de tri
    print({Sort[i]})
    for model in Models :
        tic = time.time()
        err,prtable=fc.Classifier_eq_iter(X,Y[i],model,Sort=Sort[i])
        print(f'ERR_{model}_activ_100 :',err)
        print(prtable)
        print(f'time {model}_activ_100 :',time.time()-tic)

{'classic'}
ERR_LogisticRegression(max_iter=1000000000)_activ_100 : 0.24454545454545454
   Precision     Recall   F1_Score
A  71.689944  97.284211  82.489666
C  91.865079  86.695238  88.343717
N  66.833333  13.053571  21.271212
time LogisticRegression(max_iter=1000000000)_activ_100 : 1442.7374920845032
ERR_DecisionTreeClassifier(criterion='entropy')_activ_100 : 0.3442424242424241
   Precision     Recall   F1_Score
A  71.894777  75.157895  73.176930
C  56.127778  51.380952  51.324296
N  59.407712  53.125000  54.911811
time DecisionTreeClassifier(criterion='entropy')_activ_100 : 2269.472621202469
ERR_RandomForestClassifier()_activ_100 : 0.1733333333333333
   Precision     Recall   F1_Score
A  78.847931  97.055263  86.847132
C  98.916667  58.966667  71.958514
N  91.377381  65.285714  74.840882
time RandomForestClassifier()_activ_100 : 717.230742931366
ERR_GaussianNB()_activ_100 : 0.4109090909090908
   Precision      Recall   F1_Score
A    58.5201  100.000000  73.824131
C    31.0000    5.2

## Peak classification

In [9]:
T=1/128
t=np.arange(0,len(signaux[0])*T,T)
t_activ_arr=[list(t[fc.find_peak(signal,0.7)]) for signal in signaux[:ARR]]
t_activ_chf=[list(t[fc.find_peak(signal,0.5)]) for signal in signaux[ARR:CHF]]
t_activ_nsr=[list(t[fc.find_peak(signal,2.1)]) for signal in signaux[CHF:]]
X_peak=[]
X_peak=t_activ_arr+t_activ_chf+t_activ_nsr
for x in X_peak :
    max=np.max([len(x) for x in X_peak])
    while len(x)<max :
        x.append(0)

In [10]:
for i in range(len(Y)) :# On classifie selon chacun des modes de tri
    print({Sort[i]})
    for model in Models :# classification with the signals' peak
        tic = time.time()
        err,prtable=fc.Classifier_eq_iter(X_peak,Y[i],model,Sort=Sort[i])
        print(f'ERR_{model}_peak_100 :',err)
        print(prtable)
        print(f'time {model}_peak_100 :',time.time()-tic)

{'classic'}
ERR_LogisticRegression(max_iter=1000000000)_peak_100 : 0.44696969696969696
   Precision     Recall   F1_Score
A  70.429844  56.976316  62.463810
C  30.213817  30.619048  29.298353
N  49.738016  69.821429  57.335781
time LogisticRegression(max_iter=1000000000)_peak_100 : 4738.642642021179
ERR_DecisionTreeClassifier(criterion='entropy')_peak_100 : 0.4369696969696968
   Precision     Recall   F1_Score
A  68.400580  57.402632  61.924071
C  37.485317  33.076190  33.777889
N  49.564921  71.464286  57.912737
time DecisionTreeClassifier(criterion='entropy')_peak_100 : 50.790664196014404
ERR_RandomForestClassifier()_peak_100 : 0.37909090909090915
   Precision     Recall   F1_Score
A  69.755384  70.644737  69.857537
C  33.838095  21.061905  24.982223
N  55.962471  72.910714  62.655609
time RandomForestClassifier()_peak_100 : 85.78137731552124
ERR_GaussianNB()_peak_100 : 0.6269696969696971
   Precision     Recall   F1_Score
A  75.926533  19.547368  29.652265
C  54.200000  16.995238  2

## Features Classification 

In [13]:
features=fc.extract_features(signaux)
for i in range(len(Y)) :# On classifie selon chacun des modes de tri
    print({Sort[i]})
    for model in Models :# classification with the signals' features
        tic = time.time()
        err,prtable=fc.Classifier_eq_iter(features,Y[i],model,Sort=Sort[i])
        print(f'ERR_{model}_features_100 :',err)
        print(prtable)
        print(f'time {model}_features_100 :',time.time()-tic)

{'classic'}
ERR_LogisticRegression(max_iter=1000000000)_features_100 : 0.2918181818181818
   Precision     Recall   F1_Score
A  70.987266  85.707895  77.365912
C  14.816667   5.090476   7.104135
N  78.975375  85.375000  81.027103
time LogisticRegression(max_iter=1000000000)_features_100 : 29.570689916610718
ERR_DecisionTreeClassifier(criterion='entropy')_features_100 : 0.16151515151515153
   Precision     Recall   F1_Score
A  86.740380  86.052632  86.069212
C  68.830159  69.404762  67.520159
N  92.179149  89.500000  90.091555
time DecisionTreeClassifier(criterion='entropy')_features_100 : 0.7417356967926025
ERR_RandomForestClassifier()_features_100 : 0.15909090909090914
   Precision     Recall   F1_Score
A  82.969775  92.373684  87.168288
C  76.436111  54.414286  61.007687
N  96.084776  87.214286  90.720607
time RandomForestClassifier()_features_100 : 32.14016318321228
ERR_GaussianNB()_features_100 : 0.5545454545454546
   Precision     Recall   F1_Score
A  59.321545  37.400000  45.3919

## Reconstruct signals Classification

In [15]:
z_arr,z_chf,z_nsr=Z[0],Z[1],Z[2]
d_arr,d_chf,d_nsr=D[0],D[1],D[2]
signaux_nsr_reconstruits = fc.reconstruire_signaux(z_nsr, d_nsr)
signaux_arr_reconstruits = fc.reconstruire_signaux(z_arr, d_arr)
signaux_chf_reconstruits = fc.reconstruire_signaux(z_chf, d_chf)
X_reconstruct=np.concatenate((signaux_arr_reconstruits,signaux_chf_reconstruits,signaux_nsr_reconstruits),axis=0)
for i in range(len(Y)) :# On classifie selon chacun des modes de tri
    print({Sort[i]})
    for model in Models :# classification with the signals' features
        tic = time.time()
        err,prtable=fc.Classifier_eq_iter(X_reconstruct,Y[i],model,Sort=Sort[i])
        print(f'ERR_{model}_reconstruct_100 :',err)
        print(prtable)
        print(f'time {model}_reconstruct_100 :',time.time()-tic)

{'classic'}
ERR_LogisticRegression(max_iter=1000000000)_reconstruct_100 : 0.4072727272727273
   Precision     Recall   F1_Score
A  61.995826  78.205263  68.965636
C  52.219048  28.638095  35.210379
N  51.812049  36.392857  41.184708
time LogisticRegression(max_iter=1000000000)_reconstruct_100 : 483.0042951107025
ERR_DecisionTreeClassifier(criterion='entropy')_reconstruct_100 : 0.33818181818181814
   Precision     Recall   F1_Score
A  74.571801  74.710526  74.230872
C  53.438420  48.023810  48.097379
N  59.986153  59.125000  58.009188
time DecisionTreeClassifier(criterion='entropy')_reconstruct_100 : 953.808397769928
ERR_RandomForestClassifier()_reconstruct_100 : 0.28090909090909094
   Precision     Recall   F1_Score
A  68.064172  98.644737  80.456445
C  20.500000   4.019048   6.554762
N  96.254365  58.303571  70.298092
time RandomForestClassifier()_reconstruct_100 : 179.5438575744629
ERR_GaussianNB()_reconstruct_100 : 0.4042424242424241
   Precision     Recall   F1_Score
A  64.546425  